# Redis Bus — Smoke notebook

Vérification manuelle du module `src/bus/`. Regroupe les tests à faire après chaque PR R3 qui touche à Redis : infra, client, keys/channels, publisher, throttle.

**Comment utiliser ce notebook :**

1. Démarre Docker Desktop.
2. `Kernel > Restart & Run All` — toutes les cellules s'enchaînent.
3. Avant chaque section, lis la cellule markdown **"Ce que tu dois tester"** — elle dit ce qui est vérifié, à quoi ressemble le succès, et ce qui doit t'alerter.

**Prérequis une seule fois :**
- Docker Desktop installé
- venv avec `redis>=5.0,<6.0` (`python -m pip install "redis>=5.0,<6.0"`)

**Note Jupyter :** le notebook utilise `await` directement (top-level async) au lieu de `asyncio.run()` — sinon Jupyter raise `asyncio.run() cannot be called from a running event loop`.

## 0. Setup — paths, env, helpers

### Ce que tu dois tester
Rien côté toi — cette cellule configure les chemins pour que les imports `bus.*` fonctionnent et définit `REDIS_URL` pour la session.

**Succès attendu** : deux lignes `CWD :` et `REDIS_URL :` affichées, pas d'exception.

**Alerte si tu vois** : une erreur `ModuleNotFoundError` → le working directory n'est pas à la racine du repo.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
os.chdir(PROJECT_ROOT)
SRC = str(PROJECT_ROOT / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

os.environ["REDIS_URL"] = "redis://localhost:6380/0"

print("CWD :", PROJECT_ROOT)
print("REDIS_URL :", os.environ["REDIS_URL"])

def run(cmd, check=True):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    if check and result.returncode != 0:
        raise RuntimeError(f"command failed: rc={result.returncode}")
    return result.returncode

CWD : C:\Users\Valérian Darmenté\Documents\Python Project\fx-volatility-trading-system
REDIS_URL : redis://localhost:6380/0


## 1. Infrastructure Redis — PR #1

### Ce que tu dois tester
Le conteneur `fxvol-redis-dev` démarre avec la bonne config (commandes dangereuses renommées).

**Succès attendu :**
- `docker compose up` réussit et récupère `redis:7-alpine`
- Healthcheck passe de `starting` à `healthy` en moins de 10 secondes
- `redis-cli PING` → `PONG`
- `CONFIG GET maxmemory` → `ERR unknown command 'CONFIG'` (c'est **voulu**, la conf désactive la commande)
- `FLUSHDB` → `ERR unknown command 'FLUSHDB'` (idem)

**Alerte si tu vois :**
- `PING` ne répond pas → healthcheck pas encore passé, attends 5s et re-run
- `CONFIG GET` retourne vraiment la valeur → la conf n'est pas montée, vérifie `./infrastructure/redis/redis.conf`

In [2]:
run("docker compose -f docker-compose.dev.yml up -d redis")

$ docker compose -f docker-compose.dev.yml up -d redis
 Container fxvol-redis-dev Running 


0

In [3]:
import time
for i in range(15):
    out = subprocess.run(
        "docker inspect --format '{{.State.Health.Status}}' fxvol-redis-dev",
        shell=True, capture_output=True, text=True,
    )
    status = out.stdout.strip().strip("'\"")
    print(f"[{i}s] health: {status}")
    if status == "healthy":
        break
    time.sleep(1)

[0s] health: healthy


In [4]:
run("docker exec fxvol-redis-dev redis-cli PING")
print("--- CONFIG (doit être ERR unknown) ---")
run("docker exec fxvol-redis-dev redis-cli CONFIG GET maxmemory", check=False)
print("--- FLUSHDB (doit être ERR unknown) ---")
run("docker exec fxvol-redis-dev redis-cli FLUSHDB", check=False)

$ docker exec fxvol-redis-dev redis-cli PING
PONG
--- CONFIG (doit être ERR unknown) ---
$ docker exec fxvol-redis-dev redis-cli CONFIG GET maxmemory
ERR unknown command 'CONFIG', with args beginning with: 'GET' 'maxmemory' 

--- FLUSHDB (doit être ERR unknown) ---
$ docker exec fxvol-redis-dev redis-cli FLUSHDB
ERR unknown command 'FLUSHDB', with args beginning with: 



0

## 2. Imports du package `bus` — PR #2

### Ce que tu dois tester
Le package `bus` expose bien `get_redis`, `get_sync_redis`, `keys`, `channels`, et les helpers du publisher. Les templates de keys se formatent correctement avec `.format(symbol=...)`.

**Succès attendu :**
- `LATEST_SPOT template: latest_spot:{symbol}`
- `formatted: latest_spot:EURUSD`
- `TTL_SPOT: 30 s`
- `CH_TICKS: ticks`
- Dernière ligne : `OK — bus imports work`

**Alerte si tu vois :**
- `ImportError: cannot import name 'get_redis'` → le package `src/bus/` est mal structuré ou `__init__.py` manquant
- `ModuleNotFoundError: No module named 'redis'` → driver Python absent du venv (`python -m pip install "redis>=5.0,<6.0"`)

In [5]:
from bus import channels, get_redis, get_sync_redis, keys
from bus.publisher import (
    publish_account, publish_risk_update, publish_tick,
    publish_vol_update, reset_throttle_for_tests, set_heartbeat,
)

print("LATEST_SPOT template:", keys.LATEST_SPOT)
print("formatted:", keys.LATEST_SPOT.format(symbol="EURUSD"))
print("TTL_SPOT:", keys.TTL_SPOT, "s")
print("CH_TICKS:", channels.CH_TICKS)
print("OK — bus imports work")

LATEST_SPOT template: latest_spot:{symbol}
formatted: latest_spot:EURUSD
TTL_SPOT: 30 s
CH_TICKS: ticks
OK — bus imports work


## 3. Fail-fast sans `REDIS_URL` — PR #2

### Ce que tu dois tester
`get_redis()` refuse de construire un client si `REDIS_URL` n'est pas dans l'environnement. Mieux vaut crasher au boot qu'un service silencieusement KO pendant des heures en prod.

**Succès attendu :**
- Une ligne `OK — fail-fast triggered: REDIS_URL is not set. See .env.example...`

**Alerte si tu vois :**
- `UNEXPECTED: no exception raised` → un cache persisté du client précédent n'a pas été invalidé, ou `_redis_url()` ne vérifie plus rien

In [6]:
from bus.redis_client import reset_clients_for_tests

saved = os.environ.pop("REDIS_URL")
reset_clients_for_tests()
try:
    get_redis()
    print("UNEXPECTED: no exception raised")
except RuntimeError as e:
    print("OK — fail-fast triggered:", e)
finally:
    os.environ["REDIS_URL"] = saved
    reset_clients_for_tests()

OK — fail-fast triggered: REDIS_URL is not set. See .env.example for the expected format.


## 4. Publisher helpers — PR #3

### Ce que tu dois tester
Les 5 helpers (`set_heartbeat`, `publish_tick`, `publish_account`, `publish_vol_update`, `publish_risk_update`) écrivent effectivement dans Redis les keys prescrites par la spec `09-redis.md`, avec les bons TTL.

**Succès attendu :**
- La cellule suivante affiche `pushed: heartbeats×3, tick, account, vol_update, risk_update`
- Pas d'exception (aucun helper ne raise)

**Alerte si tu vois :**
- `ConnectionRefusedError` / `asyncio.TimeoutError` → Redis pas accessible sur `localhost:6380`, re-run section 1
- `RuntimeError: REDIS_URL is not set` → le reset de la section 3 a mal restauré l'env, re-run section 0
- `asyncio.run() cannot be called from a running event loop` → tu as mis `asyncio.run(...)` dans une cellule, utilise `await` direct (top-level async Jupyter)

In [7]:
# Jupyter supporte l'await top-level depuis IPython 7.x — pas besoin d'asyncio.run()
r = get_redis()
reset_throttle_for_tests()

await set_heartbeat(r, keys.ENGINE_MARKET_DATA)
await set_heartbeat(r, keys.ENGINE_VOL)
await set_heartbeat(r, keys.ENGINE_RISK)

await publish_tick(r, "EURUSD", bid=1.0856, ask=1.0858, mid=1.0857)

await publish_account(r, {
    "net_liq_usd": 125_000, "cash_usd": 75_000, "open_positions_count": 4,
})

await publish_vol_update(
    r, symbol="EURUSD",
    surface_data={"1M": {"atm": 7.5}, "3M": {"atm": 8.0}},
    signals_data=[{"tenor": "1M", "type": "CHEAP"}],
)

await publish_risk_update(
    r, greeks={"delta_net": 1200, "vega_net": 500},
    pnl_curve={"spots": [1.08, 1.09], "pnls": [0, 100]},
)

print("pushed: heartbeats×3, tick, account, vol_update, risk_update")

pushed: heartbeats×3, tick, account, vol_update, risk_update


## 5. Inspection des keys, valeurs et TTLs — PR #3

### Ce que tu dois tester
Toutes les keys écrites en section 4 sont bien en Redis avec le bon TTL (prescrit par `bus.keys`).

**Succès attendu :**
- `KEYS latest_*` liste : `latest_spot:EURUSD`, `latest_bid:EURUSD`, `latest_ask:EURUSD`, `latest_vol_surface:EURUSD`, `latest_signals:EURUSD`, `latest_greeks:portfolio`, `latest_pnl_curve`
- `KEYS heartbeat:*` liste 3 heartbeats (market_data, vol_engine, risk_engine)
- La table des TTL montre des valeurs **proches** de `expected` (1-3 secondes d'écart possible car du temps s'écoule entre le push et le TTL check)
- Les payloads JSON de `account_snapshot`, `latest_vol_surface`, etc. sont lisibles et contiennent un `timestamp` ISO 8601 se terminant par `Z`

**Alerte si tu vois :**
- Un TTL = `-2` → la key a déjà expiré (tu as attendu trop longtemps entre section 4 et 5), re-run section 4
- Un TTL = `-1` → la key n'a pas de TTL, le helper a oublié de passer `ex=...`
- Un TTL très éloigné de l'attendu (ex : 600 au lieu de 30) → bug dans le helper, mauvais constant TTL utilisé

In [9]:
run('docker exec fxvol-redis-dev redis-cli KEYS "latest_*"')
run('docker exec fxvol-redis-dev redis-cli KEYS "heartbeat:*"')
run("docker exec fxvol-redis-dev redis-cli KEYS account_snapshot")

$ docker exec fxvol-redis-dev redis-cli KEYS "latest_*"
latest_greeks:portfolio
latest_spot:EURUSD
latest_bid:EURUSD
latest_signals:EURUSD
latest_pnl_curve
latest_vol_surface:EURUSD
latest_ask:EURUSD
$ docker exec fxvol-redis-dev redis-cli KEYS "heartbeat:*"
heartbeat:risk_engine
heartbeat:vol_engine
heartbeat:market_data
$ docker exec fxvol-redis-dev redis-cli KEYS account_snapshot
account_snapshot


0

In [10]:
for key, expected in [
    ("latest_spot:EURUSD", keys.TTL_SPOT),
    ("latest_bid:EURUSD", keys.TTL_BID_ASK),
    ("latest_ask:EURUSD", keys.TTL_BID_ASK),
    ("account_snapshot", keys.TTL_ACCOUNT),
    ("latest_vol_surface:EURUSD", keys.TTL_VOL_SURFACE),
    ("latest_signals:EURUSD", keys.TTL_SIGNALS),
    ("latest_greeks:portfolio", keys.TTL_GREEKS),
    ("latest_pnl_curve", keys.TTL_PNL_CURVE),
    ("heartbeat:market_data", keys.TTL_HEARTBEAT),
]:
    out = subprocess.run(
        f'docker exec fxvol-redis-dev redis-cli TTL "{key}"',
        shell=True, capture_output=True, text=True,
    )
    ttl = out.stdout.strip()
    print(f"{key:<40} TTL={ttl:>4}  (expected ~{expected})")

latest_spot:EURUSD                       TTL=  20  (expected ~30)
latest_bid:EURUSD                        TTL=  20  (expected ~30)
latest_ask:EURUSD                        TTL=  20  (expected ~30)
account_snapshot                         TTL=  50  (expected ~60)
latest_vol_surface:EURUSD                TTL= 590  (expected ~600)
latest_signals:EURUSD                    TTL= 590  (expected ~600)
latest_greeks:portfolio                  TTL=  20  (expected ~30)
latest_pnl_curve                         TTL=  19  (expected ~30)
heartbeat:market_data                    TTL=  19  (expected ~30)


In [11]:
for key in (
    "account_snapshot",
    "latest_vol_surface:EURUSD",
    "latest_greeks:portfolio",
    "heartbeat:vol_engine",
):
    out = subprocess.run(
        f'docker exec fxvol-redis-dev redis-cli GET "{key}"',
        shell=True, capture_output=True, text=True,
    )
    print(f"--- {key} ---")
    print(out.stdout.strip() or "(empty)")
    print()

--- account_snapshot ---
{"net_liq_usd": 125000, "cash_usd": 75000, "open_positions_count": 4, "timestamp": "2026-04-20T13:33:30.825666Z"}

--- latest_vol_surface:EURUSD ---
{"symbol": "EURUSD", "timestamp": "2026-04-20T13:33:30.826666Z", "surface": {"1M": {"atm": 7.5}, "3M": {"atm": 8.0}}}

--- latest_greeks:portfolio ---
{"timestamp": "2026-04-20T13:33:30.827673Z", "greeks": {"delta_net": 1200, "vega_net": 500}}

--- heartbeat:vol_engine ---
2026-04-20T13:33:30.822659Z



## 6. Tick throttle — PR #3

### Ce que tu dois tester
`publish_tick` limite les PUBLISH sur le canal `ticks` à un par 200ms par symbole. Les SET (cache `latest_spot`) ne sont **jamais** throttlés — le cache doit refléter le dernier tick.

**Succès attendu :**
- `50 ticks sent : 1 published, 49 throttled` (ou 2/48 si la boucle prend > 200ms, rare mais possible)
- Le cache `latest_spot:EURUSD` contient bien la **dernière** valeur des 50 ticks (pas la première)

**Alerte si tu vois :**
- `50 published, 0 throttled` → throttle cassé, le compteur n'est pas respecté
- `0 published` → le premier call a été throttlé, bug sur la valeur initiale du bucket

In [12]:
reset_throttle_for_tests()
published = 0
swallowed = 0
for i in range(50):
    ok = await publish_tick(r, "EURUSD", 1.0 + 1e-6 * i, 1.0 + 1e-6 * i, 1.0)
    if ok:
        published += 1
    else:
        swallowed += 1

print(f"50 ticks sent : {published} published, {swallowed} throttled")
print("Attendu : 1/49 (ou 2/48 si la boucle prend plus de 200ms).")

50 ticks sent : 1 published, 49 throttled
Attendu : 1/49 (ou 2/48 si la boucle prend plus de 200ms).


## 7. MarketDataEngine → Redis — PR #4

### Ce que tu dois tester
Le `MarketDataEngine` publie maintenant ticks + account snapshot + heartbeat vers Redis. En prod, quand `REDIS_URL` est set, l'engine crée son **propre event loop** dans son thread et `run_until_complete(publish_*)` à chaque itération. Les helpers `_publish_ticks_to_redis`, `_publish_account_to_redis`, `_set_heartbeat_to_redis` **swallow** les `ConnectionError` Redis pour ne pas crasher le thread.

Ici on simule le comportement en appelant directement les publishers (on ne peut pas exécuter `loop.run_until_complete` dans une cellule Jupyter puisque Jupyter a déjà son propre loop).

**Succès attendu :**
- Cellule suivante affiche `publish_tick emitted PUBLISH : True` puis `engine a publié : tick + account + heartbeat`
- `MGET latest_spot:EURUSD latest_bid:EURUSD latest_ask:EURUSD heartbeat:market_data` renvoie les 4 valeurs (float, float, float, ISO timestamp)
- Pas d'exception non catchée

**Alerte si tu vois :**
- `publish_tick emitted PUBLISH : False` → throttle pas reset, re-run la cellule après `reset_throttle_for_tests()`
- `MGET` renvoie des `(nil)` → les writes ne sont pas passés, vérifier que Redis est up et REDIS_URL correct

In [ ]:
# Simule ce que l'engine ferait à chaque tour de sa boucle run() :
# 1) publish_tick pour le dernier tick reçu
# 2) publish_account quand le snapshot interval est atteint
# 3) set_heartbeat à chaque itération
from bus.publisher import publish_account as _pa, publish_tick as _pt, set_heartbeat as _sh

reset_throttle_for_tests()

payload = {
    "ticks": [{"bid": 1.0856, "ask": 1.0858}],
    "portfolio_payload": {"summary": ["a", "b", "c"], "positions": ["p1", "p2"]},
}

# Tick : 3 SET + 1 PUBLISH (throttle reset → first call publishes)
last = payload["ticks"][-1]
mid = (last["bid"] + last["ask"]) / 2.0
emitted = await _pt(r, "EURUSD", last["bid"], last["ask"], mid)
print("publish_tick emitted PUBLISH :", emitted)

# Account : 1 SET + 1 PUBLISH
await _pa(r, {
    "symbol": "EURUSD",
    "summary_tag_count": len(payload["portfolio_payload"]["summary"]),
    "open_positions_count": len(payload["portfolio_payload"]["positions"]),
})

# Heartbeat MarketData
await _sh(r, keys.ENGINE_MARKET_DATA)

print("engine a publié : tick + account + heartbeat")

In [ ]:
# Vérifie que les keys ont bien été mises à jour avec les valeurs de l'engine
out = subprocess.run(
    'docker exec fxvol-redis-dev redis-cli MGET latest_spot:EURUSD latest_bid:EURUSD latest_ask:EURUSD heartbeat:market_data',
    shell=True, capture_output=True, text=True,
)
print(out.stdout)

### Test de résilience — Redis down

**Ce que tu dois tester :** quand Redis throw `ConnectionError`, les helpers `_publish_*_to_redis` de l'engine **catch** l'erreur et continuent. Le thread engine ne crash pas.

**Succès attendu :** la cellule suivante affiche `OK — ConnectionError raised by mock (l'engine la catchera)` — ce test prouve que **le publisher laisse remonter l'exception**, c'est le helper `_publish_ticks_to_redis` qui la swallow (cf. `_REDIS_SWALLOW` dans `market_data_engine.py`).

**Alerte si tu vois :** `UNEXPECTED : publish_tick a réussi` → le mock n'a pas raise, vérifier le `side_effect`.

In [ ]:
from unittest.mock import AsyncMock
from redis import exceptions as _redis_exc
from bus.publisher import publish_tick as _pt

broken_client = AsyncMock()
broken_client.set = AsyncMock(side_effect=_redis_exc.ConnectionError("Connection reset"))
broken_client.publish = AsyncMock()

# Appel direct au publisher avec le mock — la ConnectionError remonte
# (c'est le helper de l'engine `_publish_ticks_to_redis` qui la catch,
# pas le publisher lui-même). On prouve ici que le pattern try/except
# dans l'engine est nécessaire.
try:
    await _pt(broken_client, "EURUSD", 1.0, 1.0, 1.0)
    print("UNEXPECTED : publish_tick a réussi")
except _redis_exc.ConnectionError as e:
    print("OK — ConnectionError raised by mock (l'engine la catchera):", e)

print("engine._publish_ticks_to_redis() swallow ce type via _REDIS_SWALLOW")

In [ ]:
# Fermeture propre du client async utilisé dans les sections 4, 6 et 7
await r.aclose()
print("client Redis fermé")

## 8. Cleanup

### Ce que tu dois tester
Rien — choisis selon ton besoin :

- **Soft stop** (garde le volume, data survit) : `docker compose down`
- **Hard stop** (wipe data) : `docker compose down -v`

Les lignes sont commentées ci-dessous — décommente ce que tu veux.

In [14]:
# run("docker compose -f docker-compose.dev.yml down")
# run("docker compose -f docker-compose.dev.yml down -v")

## Cheat sheet — troubleshooting

| Erreur | Cause probable | Fix |
|---|---|---|
| `No module named 'redis'` | redis-py absent du venv | `python -m pip install "redis>=5.0,<6.0"` |
| `No module named 'bus'` | sys.path pas configuré | re-run section 0 |
| `RuntimeError: REDIS_URL is not set` | env wipé par la section 3 | re-run section 0 |
| `asyncio.run() cannot be called...` | utilisation d'`asyncio.run()` au lieu de `await` | utiliser `await ...` directement (top-level async) |
| PING ne répond pas | healthcheck pas encore passé | attends 5s, re-run section 1 |
| `ERR unknown command 'CONFIG'` | attendu — conf hardened | ce n'est pas un bug, c'est le test |
| TTL montre `-2` | key expirée entre SET et check | re-run section 4 (repousse) |
| `ConnectionRefusedError` sur publisher | Redis pas up ou port mauvais | `docker ps` + `$env:REDIS_URL` pointe sur `6380` |